# 준비

In [12]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


### 준비

In [13]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
print(torch.__version__)

# 실행할때마다 같은 결과가 나오도록. sklearn에서 random_state=42로 한 이유와 같음
torch.manual_seed(42) 

2.15.0+cpu


### 1. 데이터 선택

In [14]:
df = pd.read_csv("sentiment_long.csv", encoding="utf-8")
# 머신러닝에서는 리스트로 특징을 넘겨 줌. 왜? 특징을 사람이 찾았으니까
# 딥러닝에서는 특징이 아닌 데이터를 넘겨줌. 왜? 데이터에서 특징을 찾아야 하니까
X = df["sentence"]
y = df["label"]

### 2. 사전 준비

In [15]:
# 문장 => 숫자 
PAD = 0 # 빈칸 채우기용
UNK = 1 # 사전에 없는 글자
MAX_LEN = 100 # 문장 길이를 20글자로 맞춤(넘으면 자르고, 부족하면 채움)

# 글자 사전 생성
def build_vocab(sentences):
    """학습용 문장에 나온 글자마다 번호를 붙인다"""
    vocab = {"<PAD>": PAD, "<UNK>": UNK}
    for sentence in sentences:
        # "복선 회수가 깔끔하고 세트와 의상이 정성스럽고 그러나 결국에는 실망스러운 완성도에 화가 날 정도다."
        for ch in sentence:
            if ch not in vocab:
                vocab[ch] = len(vocab)
    return vocab
 
# 문장을 글자 사전을 통해 숫자로 만듬
def encode(sentence, vocab):
    """문장 하나 → 길이 MAX_LEN인 번호 목록"""
    ids = [vocab.get(ch, UNK) for ch in sentence][:MAX_LEN]
    # 빈칸은 앞쪽에 채운다. RNN은 마지막 글자를 읽은 뒤의 기억으로 판단하는데,
    # 뒤쪽에 빈칸이 있으면 마지막에 빈칸만 잔뜩 읽고 끝나게 된다.
    return [PAD] * (MAX_LEN - len(ids)) + ids
    


### 3. 훈련, 테스트 분리

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 4. 학습

In [17]:
# 학습 데이터로 단어 사정 생성
vocab = build_vocab(X_train)
# 단어 사전 길이 조회
print(len(vocab))
# 문장을 인코딩 해서 텐서로 변환
# 텐서 : 파이토치용 numpy배열
X_train_ids = torch.tensor([encode(s, vocab) for s in X_train])
X_test_ids = torch.tensor([encode(s, vocab) for s in X_test])
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32)

# 학습 모델
class SentimentRNN(nn.Module):
    # vocab_size
    def __init__(self, vocab_size):
        super().__init__()
        # 임베딩: 글자 번호를 숫자 16개짜리 벡터로 바꾼다 (글자의 "뜻"을 담는 숫자, 학습으로 채운다)
        self.embedding = nn.Embedding(vocab_size, 16, padding_idx=PAD)
        # RNN: 글자를 하나씩 읽으며 기억(숫자 32개)을 고쳐 쓴다
        # input_size=16은 위에서 한 글자가 16개의 숫자로 임베딩되기 때문에 한글자는 16개의 입력
        # hidden_size=32는 기억 크기로 32개를 기억
        # batch_first=True.(문장수, 글자수, 16) 순서로 입력 모양을 씀. 기본은(글자수,문장수,16)
        self.rnn = nn.RNN(input_size=16, hidden_size=32, batch_first=True)
        # self.rnn = nn.LSTM(input_size=16, hidden_size=32, batch_first=True)   # RNN → LSTM
        # 출력층: 마지막 기억 → 점수 1개
        # 기억한 32개를 이용해서 결과가 1개로 나타남
        self.fc = nn.Linear(32, 1)
 
    def forward(self, x):
        x = self.embedding(x)        # (문장 수, 20글자) → (문장 수, 20글자, 16)
        _, h = self.rnn(x)           # h: 마지막 글자까지 읽은 뒤의 기억 (1, 문장 수, 32)
        # _, (h, c) = self.rnn(x)   # h: 꺼내 쓴 기억, c: 노트(셀 상태)
        return self.fc(h[0]).squeeze(1)   # (문장 수,) 점수. 클수록 긍정
 
# 모델 생성
model = SentimentRNN(len(vocab))
print(model)

# 손실함수 선택. 학습률 그림에서 손실을 계산할 때 어떻게 계산하는지를 선택
loss_fn = nn.BCEWithLogitsLoss()   # 둘 중 하나 고르기(이진 분류)용 손실. sigmoid도 이 안에서 한다
# 최적화 도구 선택. 
# 학습률 그림에서 손실을 최소화하도록 가중치를 조절해야 하는데 조절하는 방법을 선택
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 100번의 학습을 진행
for epoch in range(1, 201):
    model.train()
    pred = model(X_train_ids)             # 1. 순전파
    loss = loss_fn(pred, y_train_t)       # 2. 손실
 
    optimizer.zero_grad()
    loss.backward()                       # 3. 역전파
    optimizer.step()                      # 4. 수정
 
    if epoch % 10 == 0: # 10번 작업할때마다 손실룰을 출력
        print(f"에포크 {epoch:3}: 손실 {loss.item():.4f}")

260
SentimentRNN(
  (embedding): Embedding(260, 16, padding_idx=0)
  (rnn): RNN(16, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)
에포크  10: 손실 0.4833
에포크  20: 손실 0.2182
에포크  30: 손실 0.1465
에포크  40: 손실 0.1829
에포크  50: 손실 0.2593
에포크  60: 손실 0.1544
에포크  70: 손실 0.0977
에포크  80: 손실 0.0525
에포크  90: 손실 0.0126
에포크 100: 손실 0.0056
에포크 110: 손실 0.0024
에포크 120: 손실 0.0015
에포크 130: 손실 0.0010
에포크 140: 손실 0.0008
에포크 150: 손실 0.0007
에포크 160: 손실 0.0006
에포크 170: 손실 0.0005
에포크 180: 손실 0.0005
에포크 190: 손실 0.0004
에포크 200: 손실 0.0004


### 평가

In [18]:
# 테스트 데이터로 확인
def accuracy(X_ids, y_t):
    model.eval()
    with torch.no_grad():
        # 활성화 함수(시그모이드) 결과를 0~1사의 값으로 반환
        prob = torch.sigmoid(model(X_ids))   # 점수 → 0~1 확률
        pred = (prob >= 0.5).float()         # 0.5 이상이면 긍정
    return (pred == y_t).float().mean().item()
 
 
print(f"학습 정확도: {accuracy(X_train_ids, y_train_t):.1%}")
print(f"시험 정확도: {accuracy(X_test_ids, y_test_t):.1%}")

학습 정확도: 100.0%
시험 정확도: 99.0%


### 예측

In [21]:
def predict(sentence):
    """새 문장도 학습 때와 같은 사전 준비(같은 사전, 같은 길이)를 거친다"""
    ids = torch.tensor([encode(sentence, vocab)])
    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(ids)).item()
    return prob
 
 
new_sentences = [
    "주연들의 연기가 탄탄하며 재미 있었고 다시 보고 싶었다",
    "주연들의 연기가 탄탄하지만 억지로 감동을 강요하는 것 같았다",
    "결과적으로 최악의 영화라는 평가를 피할 수 없다 하지만 영상미가 압도적으로 화려하고 색감이 따뜻하고 아름답다",
    "영상미가 압도적으로 화려하고 색감이 따뜻하고 아름다우며 하지만 결과적으로 최악의 영화라는 평가를 피할 수 없다",
    "조연들의 연기도 탄탄하며 배우들의 몰입감이 엄청나며 그래서 결국 다시 보고 싶을 만큼 만족스러웠다.",        # '나쁘'가 있지만 긍정
    "캐릭터들이 하나같이 평면적이고 결말이 허무하기 짝이 없으며 배우들의 연기가 자꾸 거슬리고 그렇지만 다시 보고 싶을 만큼 만족스러웠다",          # '좋'이 있지만 부정
]
for s in new_sentences:
    p = predict(s)
    print(f"{s:20} → {'긍정' if p >= 0.5 else '부정'} ({p:.2f})")
 

주연들의 연기가 탄탄하며 재미 있었고 다시 보고 싶었다 → 긍정 (1.00)
주연들의 연기가 탄탄하지만 억지로 감동을 강요하는 것 같았다 → 긍정 (1.00)
결과적으로 최악의 영화라는 평가를 피할 수 없다 하지만 영상미가 압도적으로 화려하고 색감이 따뜻하고 아름답다 → 부정 (0.00)
영상미가 압도적으로 화려하고 색감이 따뜻하고 아름다우며 하지만 결과적으로 최악의 영화라는 평가를 피할 수 없다 → 부정 (0.01)
조연들의 연기도 탄탄하며 배우들의 몰입감이 엄청나며 그래서 결국 다시 보고 싶을 만큼 만족스러웠다. → 긍정 (1.00)
캐릭터들이 하나같이 평면적이고 결말이 허무하기 짝이 없으며 배우들의 연기가 자꾸 거슬리고 그렇지만 다시 보고 싶을 만큼 만족스러웠다 → 긍정 (0.98)
